# Statutory Knowledge Graph: Light-Mode Topology & Multi-Perspective Visualizations

**Thesis Title:** *A Coarse-to-Fine Semantic Conflict Detection System for Ex-Ante Davao City Ordinances Using Information Retrieval and Natural Language Inference*  
**Authors:** Ralph Paolo Dulce & Yahyah Odin (Ateneo de Davao University)  

This notebook loads the complete Canonical Statutory Knowledge Graph (**27,248 unique enactments**: **25,432** national statutes + **1,816** Davao City local ordinances, and **30,034** directed citation, amendment, and repeal edges) from output/visualizations/statutory_kg_full_web.html and generates formal light-mode, publication-grade matplotlib figures showing all 27,248 enactments across all **5 spatial layouts** and **5 color-coding views**:
1. kg_concentric_jurisdiction_overview.png (Concentric Planetary Orbits, Jurisdiction Contrast — Ch. 4 highlight)
2. kg_vertical_hierarchy_domains.png (Vertical Preemption Hierarchy + Macro Domain Constellations — Ch. 4 highlight)
3. kg_appendix_all_5_layouts.png (Complete 5-Layout Comparison Grid — Appendix)
4. kg_appendix_color_domains_instruments.png (Macro Legal Domains & Statutory Instrument Classes — Appendix)
5. kg_appendix_color_eras_typology.png (Legislative Era Evolution & Fine-Grained Corpus Typology — Appendix)


In [1]:
# Cell 1: Load Canonical Knowledge Graph Nodes (27,248) and Edges (30,034)
import os, re, json
from collections import Counter
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D

plt.rcParams.update({
    'font.sans-serif': 'DejaVu Sans',
    'axes.edgecolor': '#334155',
    'axes.linewidth': 0.85,
    'figure.facecolor': '#FFFFFF',
    'axes.facecolor': '#FAFCFE',
    'savefig.facecolor': '#FFFFFF',
})

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
HTML_PATH = os.path.join(REPO_ROOT, 'output', 'visualizations', 'statutory_kg_full_web.html')
FIGS_DIR = os.path.join(REPO_ROOT, 'CS_Undergraduate_Thesis_Template', 'figs')
os.makedirs(FIGS_DIR, exist_ok=True)

with open(HTML_PATH, 'r', encoding='utf-8') as f:
    html_text = f.read()

NODES = json.loads(re.search(r'const NODES = (\[.*?\]);', html_text, re.S).group(1))
EDGES = json.loads(re.search(r'const EDGES = (\[.*?\]);', html_text, re.S).group(1))
id_to_idx = {n['id']: i for i, n in enumerate(NODES)}

print(f'Loaded {len(NODES):,} unique enactments and {len(EDGES):,} structural edges.')
print('Jurisdiction breakdown:', dict(Counter(n['jur'] for n in NODES)))
print('Instrument breakdown:', dict(Counter(n['type'] for n in NODES)))


Loaded 27,248 unique enactments and 30,034 structural edges.
Jurisdiction breakdown: {'national': 25432, 'local': 1816}
Instrument breakdown: {'RA': 11955, 'OTHER': 362, 'ACT': 3915, 'BP': 885, 'EO': 5739, 'CA': 733, 'PD': 1843, 'ORD': 1816}


In [2]:
# Cell 2: Canonical Light-Mode Academic Palettes & Vectorized Plotting Helper
JUR_COLORS = {'national': '#B91C1C', 'local': '#0284C7'}
TYPE_COLORS = {
    'RA': '#059669', 'EO': '#DB2777', 'ACT': '#DC2626', 'PD': '#7C3AED',
    'ORD': '#0284C7', 'BP': '#D97706', 'CA': '#B45309', 'OTHER': '#64748B'
}
ERA_COLORS = {
    'Pre-War & Commonwealth (<1946)': '#B45309',
    'Third Republic & Martial Law (1946–1986)': '#DC2626',
    'Fifth Republic & Decentralization (1987–2015)': '#0284C7',
    'Modern Digital Era (2016–Present)': '#059669'
}
DOM_COLORS = {
    'Executive Issuances & Policy Reorganization': '#DB2777',
    'Social Services, Education, Labor & Human Development': '#059669',
    'General Administration, Organization & Civil Service': '#2563EB',
    'Public Works, Transportation, Utilities & Infrastructure': '#7C3AED',
    'Public Finance, Taxation, Fiscal Policy & Budget': '#D97706',
    'Substantive Regulatory, Health & Penal Ordinances': '#0891B2',
    'Inter-Agency Agreements & Institutional MOAs/MOUs': '#EA580C',
    'Environmental Protection, Zoning, Land Use & Natural Resources': '#65A30D',
}
RAW_TYPOLOGY_COLORS = {
    'Substantive Regulatory, Health & Penal Ordinances': '#0284C7',
    'Temporary Road Closures & Traffic Management': '#06B6D4',
    'Deeds of Donation & Property Usufruct': '#7C3AED',
    'Disaster Risk Reduction, Calamity & QRF Funds': '#E11D48',
    'Zoning Reclassifications & Subdivision Development': '#16A34A',
    'Inter-Agency Agreements & Institutional MOAs/MOUs': '#EA580C',
    'Executive Issuances & Policy Reorganization': '#DB2777',
    'Education & Academic Institutions': '#059669',
    'Local Government & Territorial Boundaries': '#2563EB',
    'Public Utilities & Telecommunications Franchises': '#8B5CF6',
    'Public Health, Hospitals & Medical Services': '#0D9488',
    'Statutory Codes & General Legal Amendments': '#CA8A04',
    'Public Finance & General Appropriations': '#D97706',
    'Taxation, Tariffs & Revenue Administration': '#B45309',
}

def render_kg_panel(ax, coord_key, color_array, title, draw_edges=True, edge_alpha=0.055,
                    highlight_local_on_top=True, point_scale=1.0, annotate_hubs=False,
                    aspect_adjustable='box'):
    xk, yk = 'x' + coord_key, 'y' + coord_key
    xs = np.array([n[xk] for n in NODES], dtype=np.float32)
    ys = np.array([n[yk] for n in NODES], dtype=np.float32)
    degs = np.array([n['deg'] for n in NODES], dtype=np.float32)
    is_loc = np.array([n['jur'] == 'local' for n in NODES])
    sizes = (2.2 + np.clip(np.sqrt(np.maximum(degs, 0)) * 2.1, 0, 26)) * point_scale

    if draw_edges:
        segs = []
        for e in EDGES:
            u, v = id_to_idx.get(e['u']), id_to_idx.get(e['v'])
            if u is not None and v is not None:
                segs.append(((xs[u], ys[u]), (xs[v], ys[v])))
        lc = LineCollection(segs, colors='#94A3B8', linewidths=0.22, alpha=edge_alpha, zorder=1, rasterized=True)
        ax.add_collection(lc)

    # Plot national (underneath if highlighting local) then local, then top hubs
    cols = np.asarray(color_array)
    if highlight_local_on_top:
        nat_mask = ~is_loc
        ax.scatter(xs[nat_mask], ys[nat_mask], s=sizes[nat_mask], c=cols[nat_mask],
                   alpha=0.58, linewidths=0, zorder=2, rasterized=True)
        ax.scatter(xs[is_loc], ys[is_loc], s=sizes[is_loc] * 1.45, c=cols[is_loc],
                   alpha=0.88, linewidths=0.15, edgecolors='#FFFFFF', zorder=3, rasterized=True)
    else:
        ax.scatter(xs, ys, s=sizes, c=cols, alpha=0.68, linewidths=0, zorder=2, rasterized=True)

    # Highlight top statutory hub RA 7160 with crisp academic gold star
    ra_idx = id_to_idx.get('ra_7160_1991')
    if ra_idx is not None:
        ax.scatter([xs[ra_idx]], [ys[ra_idx]], s=155 * point_scale, marker='*',
                   color='#F59E0B', edgecolors='#0F172A', linewidths=0.9, zorder=5)
        if annotate_hubs:
            span_x, span_y = (xs.max() - xs.min()), (ys.max() - ys.min())
            if coord_key == 'h':
                tx, ty = xs[ra_idx] - span_x * 0.28, ys[ra_idx] + span_y * 0.18
            elif coord_key == 'c':
                tx, ty = xs[ra_idx] + span_x * 0.14, ys[ra_idx] + span_y * 0.13
            else:
                tx, ty = xs[ra_idx] + span_x * 0.08, ys[ra_idx] - span_y * 0.11
            ax.annotate('RA 7160 (LGC 1991)\ndeg = 1,578',
                        xy=(xs[ra_idx], ys[ra_idx]),
                        xytext=(tx, ty),
                        fontsize=11.0, fontweight='bold', color='#0F172A',
                        bbox=dict(boxstyle='round,pad=0.3', fc='#FEF3C7', ec='#D97706', lw=0.9),
                        arrowprops=dict(arrowstyle='->', color='#0F172A', lw=1.0),
                        zorder=6)

    pad_x = (xs.max() - xs.min()) * 0.06
    pad_y = (ys.max() - ys.min()) * 0.06
    ax.set_xlim(xs.min() - pad_x, xs.max() + pad_x)
    # Canvas coordinates have +y downward; invert so Vertical Hierarchy shows National Top -> Local Bottom
    ax.set_ylim(ys.max() + pad_y, ys.min() - pad_y)
    ax.set_aspect('equal', adjustable=aspect_adjustable)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=12.5, fontweight='bold', pad=8, color='#0F172A')


In [3]:
# Cell 3: Figure 1 (Main Text Highlight) - Concentric Planetary Orbits by Jurisdiction
fig, ax = plt.subplots(figsize=(8.6, 8.8), dpi=300)
jur_cols = [JUR_COLORS[n['jur']] for n in NODES]

# Subtle orbital rings around RA 7160 origin (0, 0)
for radius in [1200, 2400, 3600, 4800]:
    circ = plt.Circle((0, 0), radius, color='#CBD5E1', fill=False, linestyle='--', linewidth=0.75, zorder=0)
    ax.add_patch(circ)

render_kg_panel(ax, 'c', jur_cols,
                'Canonical Statutory Knowledge Graph: Concentric Planetary Orbits (N = 27,248 Enactments)',
                draw_edges=True, edge_alpha=0.05, highlight_local_on_top=True, point_scale=1.1, annotate_hubs=False)

# Label concentric rings concisely on clear axes
for r_lbl, r_val in [('Primary Hub Orbit', 1200), ('Statute-to-Ordinance Bridge', 2400),
                     ('Intermediate Enactments', 3600), ('Outer Peripheral Rim', 4800)]:
    ax.text(0, -r_val + 160, r_lbl, ha='center', va='bottom', fontsize=10.5, color='#475569',
            bbox=dict(boxstyle='round,pad=0.18', fc='#FFFFFF', ec='none', alpha=0.82), zorder=4)

handles = [
    Line2D([0], [0], marker='o', color='w', label='National Statutes (n = 25,432; 93.34%)',
           markerfacecolor=JUR_COLORS['national'], markersize=7.5),
    Line2D([0], [0], marker='o', color='w', label='Davao City Local Ordinances (n = 1,816; 6.66%)',
           markerfacecolor=JUR_COLORS['local'], markersize=7.5),
    Line2D([0], [0], marker='*', color='w', label='RA 7160 (Local Gov. Code, 1,578 edges)',
           markerfacecolor='#F59E0B', markeredgecolor='#0F172A', markersize=11),
    Line2D([0], [0], color='#94A3B8', lw=1.2, label='Directed Citation / Amendment / Repeal Edges (30,034)'),
]
ax.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, -0.10), ncol=2,
          frameon=True, facecolor='#FFFFFF', edgecolor='#CBD5E1', fontsize=10.5)
plt.tight_layout()
out1 = os.path.join(FIGS_DIR, 'kg_concentric_jurisdiction_overview.png')
fig.savefig(out1, dpi=300, bbox_inches='tight')
plt.close(fig)
print('Saved:', out1)


Saved: C:\Users\SHRIMP\Documents\thesis-repo\CS_Undergraduate_Thesis_Template\figs\kg_concentric_jurisdiction_overview.png


In [4]:
# Cell 4: Figure 2 (Main Text Highlight) - Vertical Preemption Hierarchy & Domain Constellations
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.6, 12.6), dpi=300, gridspec_kw={'height_ratios': [1.0, 1.82]})

type_cols = [TYPE_COLORS[n['type']] for n in NODES]
render_kg_panel(ax1, 'h', type_cols, '(a) Vertical Lex Superior Hierarchy by Instrument Type',
                draw_edges=True, edge_alpha=0.045, highlight_local_on_top=True, point_scale=0.9, annotate_hubs=False)
ax1.axhline(1700, color='#475569', linestyle=':', linewidth=1.0, zorder=4)
ax1.text(-4750, 1580, 'National Preemption Strata (25,432 Statutes)', fontsize=10.2, fontweight='bold', color='#1E293B',
         bbox=dict(fc='#FFFFFF', ec='none', alpha=0.85, pad=1.5), zorder=5)
ax1.text(-4750, 1920, 'Davao City Local Ordinances (1,816)', fontsize=10.2, fontweight='bold', color='#0369A1',
         bbox=dict(fc='#FFFFFF', ec='none', alpha=0.85, pad=1.5), zorder=5)

type_counts = Counter(n['type'] for n in NODES)
h1 = [Line2D([0], [0], marker='o', color='w', label=f'{k} ({type_counts[k]:,})',
             markerfacecolor=TYPE_COLORS[k], markersize=6.5) for k in ['RA', 'EO', 'ACT', 'PD', 'ORD', 'BP', 'CA', 'OTHER']]
h1.append(Line2D([0], [0], marker='*', color='w', label='RA 7160 (LGC 1991, deg = 1,578)',
                 markerfacecolor='#F59E0B', markeredgecolor='#0F172A', markersize=10))
ax1.legend(handles=h1, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=3,
           frameon=True, facecolor='#FFFFFF', edgecolor='#CBD5E1', fontsize=9.6)

dom_cols = [DOM_COLORS.get(n['dom'], '#64748B') for n in NODES]
render_kg_panel(ax2, 'd', dom_cols, '(b) Semantic Domain Constellations (Topic Clusters)',
                draw_edges=True, edge_alpha=0.04, highlight_local_on_top=False, point_scale=0.9, annotate_hubs=False)
dom_counts = Counter(n['dom'] for n in NODES)
short_dom = {
    'Executive Issuances & Policy Reorganization': 'Exec. Issuances & Policy',
    'Social Services, Education, Labor & Human Development': 'Social Services & Education',
    'General Administration, Organization & Civil Service': 'Admin & Civil Service',
    'Public Works, Transportation, Utilities & Infrastructure': 'Public Works & Utilities',
    'Public Finance, Taxation, Fiscal Policy & Budget': 'Finance, Tax & Budget',
    'Substantive Regulatory, Health & Penal Ordinances': 'Local Regulatory & Penal',
    'Inter-Agency Agreements & Institutional MOAs/MOUs': 'Inter-Agency Agreements',
    'Environmental Protection, Zoning, Land Use & Natural Resources': 'Environment & Zoning',
}
h2 = [Line2D([0], [0], marker='o', color='w', label=f'{short_dom[k]} ({dom_counts[k]:,})',
             markerfacecolor=DOM_COLORS[k], markersize=6.5) for k in DOM_COLORS]
h2.append(Line2D([0], [0], marker='*', color='w', label='RA 7160 (LGC 1991, deg = 1,578)',
                 markerfacecolor='#F59E0B', markeredgecolor='#0F172A', markersize=10))
ax2.legend(handles=h2, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=2,
           frameon=True, facecolor='#FFFFFF', edgecolor='#CBD5E1', fontsize=9.8)

plt.tight_layout(h_pad=3.0)
out2 = os.path.join(FIGS_DIR, 'kg_vertical_hierarchy_domains.png')
fig.savefig(out2, dpi=300, bbox_inches='tight')
plt.close(fig)
print('Saved:', out2)


Saved: C:\Users\SHRIMP\Documents\thesis-repo\CS_Undergraduate_Thesis_Template\figs\kg_vertical_hierarchy_domains.png


In [5]:
# Cell 5: Figure 3 (Appendix) - Complete 5-Layout Comparison Grid (Compact 2x3 for Appendix P Page 1)
fig, axes = plt.subplots(2, 3, figsize=(11.4, 7.6), dpi=300)
jur_cols = [JUR_COLORS[n['jur']] for n in NODES]

panels = [
    ('r', '(a) Force-Directed Web'),
    ('d', '(b) Domain Constellations'),
    ('h', '(c) Vertical Hierarchy'),
    ('c', '(d) Concentric Orbits'),
    ('w', '(e) Polar Centrality Fan'),
]
for ax, (key, ttl) in zip(axes.flat[:5], panels):
    render_kg_panel(ax, key, jur_cols, ttl, draw_edges=True, edge_alpha=0.045,
                    highlight_local_on_top=True, point_scale=0.68, annotate_hubs=False,
                    aspect_adjustable='datalim')

ax_info = axes.flat[5]
ax_info.set_facecolor('#F8FAFC')
ax_info.set_xticks([])
ax_info.set_yticks([])
for sp in ax_info.spines.values():
    sp.set_edgecolor('#CBD5E1')
ax_info.set_title('(f) Graph Census & Legend', fontsize=12.0, fontweight='bold', pad=8, color='#0F172A')

summary_lines = [
    'Unique Enactments:  27,248',
    '  National: 25,432 (93.34%)',
    '  Local:     1,816  (6.66%)',
    '',
    'Directed Edges:     30,034',
    '  Citations:  24,996 (83.2%)',
    '  Amendments:  4,161 (13.9%)',
    '  Repeals:       877  (2.9%)',
    '',
    'Top Hub: RA 7160 (deg 1,578)',
    'PageRank: 0.022606',
]
ax_info.text(0.05, 0.95, '\n'.join(summary_lines), transform=ax_info.transAxes,
             va='top', ha='left', fontsize=9.6, family='monospace', color='#1E293B')

h_leg = [
    Line2D([0], [0], marker='o', color='w', label='National Statutes (25,432)',
           markerfacecolor=JUR_COLORS['national'], markersize=7.5),
    Line2D([0], [0], marker='o', color='w', label='Davao City Ordinances (1,816)',
           markerfacecolor=JUR_COLORS['local'], markersize=7.5),
    Line2D([0], [0], marker='*', color='w', label='RA 7160',
           markerfacecolor='#F59E0B', markeredgecolor='#0F172A', markersize=10),
    Line2D([0], [0], color='#94A3B8', lw=1.2, label='Directed Legal Edges (30,034)'),
]
ax_info.legend(handles=h_leg, loc='lower center', bbox_to_anchor=(0.5, 0.02), ncol=1,
               frameon=True, facecolor='#FFFFFF', edgecolor='#CBD5E1', fontsize=9.6)

plt.tight_layout(h_pad=1.0, w_pad=1.0)
out3 = os.path.join(FIGS_DIR, 'kg_appendix_all_5_layouts.png')
fig.savefig(out3, dpi=300, bbox_inches='tight')
plt.close(fig)
print('Saved:', out3)


Saved: C:\Users\SHRIMP\Documents\thesis-repo\CS_Undergraduate_Thesis_Template\figs\kg_appendix_all_5_layouts.png


In [6]:
# Cell 6: Figure 4 (Appendix) - Macro Domains & Statutory Instruments Color Views
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.2, 14.0), dpi=300)

dom_cols = [DOM_COLORS.get(n['dom'], '#64748B') for n in NODES]
render_kg_panel(ax1, 'r', dom_cols, '(a) Force-Directed Web Colored by Macro Legal Domain',
                draw_edges=True, edge_alpha=0.045, highlight_local_on_top=False, point_scale=0.88, annotate_hubs=False)
dom_counts = Counter(n['dom'] for n in NODES)
short_dom = {
    'Executive Issuances & Policy Reorganization': 'Exec. Issuances & Policy',
    'Social Services, Education, Labor & Human Development': 'Social Services & Education',
    'General Administration, Organization & Civil Service': 'Admin & Civil Service',
    'Public Works, Transportation, Utilities & Infrastructure': 'Public Works & Utilities',
    'Public Finance, Taxation, Fiscal Policy & Budget': 'Finance, Tax & Budget',
    'Substantive Regulatory, Health & Penal Ordinances': 'Local Regulatory & Penal',
    'Inter-Agency Agreements & Institutional MOAs/MOUs': 'Inter-Agency Agreements',
    'Environmental Protection, Zoning, Land Use & Natural Resources': 'Environment & Zoning',
}
h1 = [Line2D([0], [0], marker='o', color='w', label=f'{short_dom[k]} ({dom_counts[k]:,})',
             markerfacecolor=DOM_COLORS[k], markersize=6.5) for k in DOM_COLORS]
ax1.legend(handles=h1, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=2,
           frameon=True, facecolor='#FFFFFF', edgecolor='#CBD5E1', fontsize=10.0)

type_cols = [TYPE_COLORS[n['type']] for n in NODES]
render_kg_panel(ax2, 'w', type_cols, '(b) Polar Centrality Fan Colored by Instrument Type',
                draw_edges=True, edge_alpha=0.045, highlight_local_on_top=True, point_scale=0.88, annotate_hubs=False)
type_counts = Counter(n['type'] for n in NODES)
h2 = [Line2D([0], [0], marker='o', color='w', label=f'{k} ({type_counts[k]:,})',
             markerfacecolor=TYPE_COLORS[k], markersize=6.5) for k in ['RA', 'EO', 'ACT', 'PD', 'ORD', 'BP', 'CA', 'OTHER']]
ax2.legend(handles=h2, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=4,
           frameon=True, facecolor='#FFFFFF', edgecolor='#CBD5E1', fontsize=10.0)

plt.tight_layout(h_pad=3.2)
out4 = os.path.join(FIGS_DIR, 'kg_appendix_color_domains_instruments.png')
fig.savefig(out4, dpi=300, bbox_inches='tight')
plt.close(fig)
print('Saved:', out4)


Saved: C:\Users\SHRIMP\Documents\thesis-repo\CS_Undergraduate_Thesis_Template\figs\kg_appendix_color_domains_instruments.png


In [7]:
# Cell 7: Figure 5 (Appendix) - Legislative Era & Corpus-Specific Typology Color Views
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.6, 12.8), dpi=300, gridspec_kw={'height_ratios': [1.90, 1.05]})

era_cols = [ERA_COLORS.get(n['era'], '#64748B') for n in NODES]
render_kg_panel(ax1, 'w', era_cols, '(a) Polar Centrality Fan Colored by Legislative Era',
                draw_edges=True, edge_alpha=0.045, highlight_local_on_top=False, point_scale=0.92, annotate_hubs=False)
era_counts = Counter(n['era'] for n in NODES)
h1 = [Line2D([0], [0], marker='o', color='w', label=f'{k} ({era_counts[k]:,})',
             markerfacecolor=ERA_COLORS[k], markersize=7.0) for k in ERA_COLORS]
ax1.legend(handles=h1, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=2,
           frameon=True, facecolor='#FFFFFF', edgecolor='#CBD5E1', fontsize=9.4)

raw_cols = [RAW_TYPOLOGY_COLORS.get(n['raw_dom'], DOM_COLORS.get(n['dom'], '#64748B')) for n in NODES]
render_kg_panel(ax2, 'h', raw_cols, '(b) Vertical Hierarchy Colored by Fine-Grained Typology',
                draw_edges=True, edge_alpha=0.045, highlight_local_on_top=True, point_scale=0.92, annotate_hubs=False)
raw_counts = Counter(n['raw_dom'] for n in NODES)
short_raw = {
    'Executive Issuances & Policy Reorganization': 'Exec. Issuances & Policy Reorg.',
    'Education & Academic Institutions': 'Education & Academic Institutions',
    'Local Government & Territorial Boundaries': 'Local Gov. & Territorial Boundaries',
    'Public Utilities & Telecommunications Franchises': 'Public Utilities & Telecom Franchises',
    'Public Health, Hospitals & Medical Services': 'Public Health & Medical Services',
    'Statutory Codes & General Legal Amendments': 'Statutory Codes & Legal Amendments',
    'Public Finance & General Appropriations': 'Public Finance & Appropriations',
    'Substantive Regulatory, Health & Penal Ordinances': 'Local Regulatory, Health & Penal',
}
top_raw = [k for k, _ in raw_counts.most_common(8)]
h2 = [Line2D([0], [0], marker='o', color='w',
             label=f'{short_raw.get(k, k)} ({raw_counts[k]:,})',
             markerfacecolor=RAW_TYPOLOGY_COLORS.get(k, '#64748B'), markersize=7.0) for k in top_raw]
ax2.legend(handles=h2, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=2,
           frameon=True, facecolor='#FFFFFF', edgecolor='#CBD5E1', fontsize=9.4)

plt.tight_layout(h_pad=2.2)
out5 = os.path.join(FIGS_DIR, 'kg_appendix_color_eras_typology.png')
fig.savefig(out5, dpi=300, bbox_inches='tight')
plt.close(fig)
print('Saved:', out5)


Saved: C:\Users\SHRIMP\Documents\thesis-repo\CS_Undergraduate_Thesis_Template\figs\kg_appendix_color_eras_typology.png
